# A03 — Tokenizer probe

Due 5:00 AM Fri 11 September. Run this top to bottom with every assertion passing,
then write `tokenlab/FINDINGS.md`.

You are not building a tokenizer yet. You are *measuring* one, so that when you build
your own in A04 you have something to compare it against.

## Where Karpathy's code goes, and where yours goes

Three different things end up in this notebook and they do not mix.

| Section | Whose code | Graded? |
|---|---|---|
| **0. Scratch** | Karpathy's, typed along with the video | No. Never read for correctness. |
| **1–3. The probe** | Yours. Your strings, your paragraph, your numbers. | Yes |
| **4. `get_stats` and `merge`** | His algorithm, your implementation | Yes, and by `tests/test_helpers.py` in A04 |
| **5. Write it up** | Yours | Yes, via `tokenlab/FINDINGS.md` |

Section 0 is a sandbox. Follow him line for line in there, break things, leave the
mess. Nothing in it is marked and nothing below it may depend on it.

Sections 1 through 3 are not in the video at all. He does not run these probes; you do,
on strings you chose, and the numbers you get are the point of the assignment.

Section 4 is the one place his work becomes your committed code. **Type it, do not paste
it**, and do not copy his variable names if you do not know what they mean. In A04 you
port these two functions into `bpe/tokenizer.py` and a test suite checks them, so the
version that matters is the one you can explain.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # explicit name, never encoding_for_model

def pieces(s):
    """The decoded pieces, not the ids. The ids tell you nothing by eye."""
    return [enc.decode([t]) for t in enc.encode(s)]

def report(s, label=None):
    ids = enc.encode(s)
    print(f"{label or repr(s):<28} {len(ids):>3} tokens   {pieces(s)}")
    return ids


## 0. Scratch — following the video

Type along with Karpathy here. `ord()`, `chr()`, poking at `.encode("utf-8")`, his
walkthrough of building the tokenizer class: all of it belongs in this section and
nowhere else.

Add as many cells as you want. This section is not graded, is not read for correctness,
and is not allowed to be imported by anything below it. If you find yourself needing a
value from up here further down, copy it down there deliberately.

In [ ]:
# Scratch. Follow the video. Break things here.

## 1. The probe set

Every string below is here because it breaks an assumption people arrive with.
Record the count *and* the pieces for each one; `tokenlab/FINDINGS.md` wants both.

**None of this section is in the video.** These are your measurements, on strings you
picked, and nobody else's numbers will match yours.

In [ ]:
PROBES = [
    "1234567890",
    "12,345,678",
    "hello", " hello", "Hello",
    "    ", "\t",
    "    def foo():",
    "strawberry",
    "\U0001F642\U0001F643",
]

for p in PROBES:
    report(p)


## 2. The same paragraph in four languages

Replace these with a real paragraph of about 100 English words and your own
translations. The ratio between best and worst case is the number that matters,
because somebody pays per token in every one of these languages.


In [ ]:
PARA = {
    "English":  "TODO: your ~100 word paragraph",
    "Spanish":  "TODO",
    "Japanese": "TODO",
    "Arabic":   "TODO",
}

counts = {k: len(enc.encode(v)) for k, v in PARA.items()}
for k, n in counts.items():
    print(f"{k:<10} {n:>4} tokens")
print(f"\nworst/best ratio: {max(counts.values()) / min(counts.values()):.2f}x")


## 3. Bytes, not characters

The whole of A04 rests on this cell. If you build anything on `list(text)` next week
you will get a tokenizer that works until the first emoji.


In [ ]:
s = "h\u00e9llo"
print(f"{len(s)} characters, {len(s.encode('utf-8'))} bytes")
print(list(s.encode("utf-8")))

assert len(s) == 5 and len(s.encode("utf-8")) == 6, (
    "if this fails you are not looking at what you think you are looking at")


## 4. `get_stats` and `merge`

**This is Karpathy's algorithm and your implementation.** He derives both in the video;
write them here in your own hand rather than pasting from the transcript or from a
chatbot. Section 0 is where you followed along. This is where you show you understood it.

These two functions are the whole of A04's training loop. Write them today, while nothing
else is competing for your attention. Every assertion below has to pass before you start
A04, and `tests/test_helpers.py` in this repo tests exactly these two contracts.

In [ ]:
def get_stats(ids, counts=None):
    """Count adjacent pairs. get_stats([1,2,3,1,2]) -> {(1,2):2, (2,3):1, (3,1):1}"""
    # TODO
    raise NotImplementedError


def merge(ids, pair, idx):
    """Replace every consecutive `pair` with `idx`."""
    # TODO
    raise NotImplementedError


In [ ]:
assert get_stats([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
assert get_stats([]) == {}
assert get_stats([7]) == {}
assert get_stats([1, 1, 1]) == {(1, 1): 2}      # overlapping run, counted twice

c = {(9, 9): 5}
assert get_stats([1, 2], c) is c and c[(1, 2)] == 1   # accumulates in place

assert merge([1, 2, 3, 1, 2], (1, 2), 4) == [4, 3, 4]
assert merge([], (1, 2), 3) == []
assert merge([1], (1, 2), 3) == [1]
assert merge([1, 1, 1], (1, 1), 4) == [4, 1]    # do not step by one after a match
assert merge([1, 2], (1, 2), 4) == [4]

ids = [1, 2, 3]
get_stats(ids); merge(ids, (1, 2), 4)
assert ids == [1, 2, 3], "neither function may mutate its input"

print("all assertions passed — you are ready for A04")


## 5. Write it up

`tokenlab/FINDINGS.md` needs the count and the decoded pieces for every probe string,
plus **three concrete claims about the tokenizer that your data supports.** A claim
with no number next to it is not a claim.
